# Phase 5 – Important Region Mining

In [ ]:
from pathlib import Path
import itertools
import importlib.util
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import cv2
from skimage.segmentation import slic
from skimage.color import rgb2lab, rgb2hsv
from skimage.feature import local_binary_pattern, hog
from skimage.util import img_as_float
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib

DATASET_ROOT = Path("Dataset") / "Animals-10"
OUTPUT_ROOT = Path("newDataset")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SLIC_SEGMENTS = 140
SLIC_COMPACTNESS = 20.0
SLIC_SIGMA = 1.0
SUPERPIXEL_MIN_PIXELS = 1
K_RANGE = range(4, 11)
SILHOUETTE_TOL = 0.05
MIN_IMPORTANT_CLUSTERS = 3
LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_BINS = LBP_POINTS + 2

np.random.seed(42)

In [ ]:
df = pd.read_csv("feature2.csv")
if "split" not in df:
    raise ValueError("feature2.csv must include split column.")

target_col = "label"
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != target_col]

class_to_label = df.groupby("class")[target_col].first().to_dict()
label_to_class = {v: k for k, v in class_to_label.items()}

train_mask = df["split"] == "train"
X_train = df.loc[train_mask, feature_cols].astype(np.float32)
y_train = df.loc[train_mask, target_col].astype(int)

manual_bundle = joblib.load("svm_manual_scaled.pkl")
manual_model = manual_bundle["model"]

prob_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        C=manual_model.C,
        gamma=manual_model.gamma,
        kernel=manual_model.kernel,
        class_weight=manual_model.class_weight,
        probability=True,
        random_state=42,
    )),
])
prob_pipeline.fit(X_train, y_train)

In [ ]:
spec = importlib.util.spec_from_file_location("phase2_features", str(Path("phase2.py")))
phase2_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(phase2_mod)

def compute_global_features_from_bgr(img_bgr: np.ndarray) -> np.ndarray:
    if img_bgr is None:
        raise ValueError("Image is empty.")
    h, w = img_bgr.shape[:2]
    aspect_ratio = w / h if h else 0.0
    resized = cv2.resize(img_bgr, phase2_mod.PROCESS_SIZE)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

    mean_r, mean_g, mean_b, std_r, std_g, std_b = phase2_mod.compute_rgb_stats(resized)
    mean_h, mean_s, mean_v, std_h, std_s, std_v = phase2_mod.compute_hsv_stats(resized)
    (
        mean_int,
        std_int,
        min_int,
        max_int,
        range_int,
        skewness_val,
        kurtosis_val,
        entropy_val,
    ) = phase2_mod.compute_intensity_stats(gray)
    contrast, dissimilarity, homogeneity, energy, correlation, ASM = phase2_mod.compute_glcm_features(gray)
    edge_mean, edge_std, edge_density = phase2_mod.compute_edge_features(gray)
    contour_area, contour_perimeter, compactness = phase2_mod.compute_contour_features(gray)
    lbp_hist = phase2_mod.compute_lbp_features(gray)
    color_hist = phase2_mod.compute_color_histograms(resized)
    hog_feats = phase2_mod.compute_hog_features(gray)

    brightness = mean_v
    contrast_simple = std_int

    feats = [
        aspect_ratio,
        mean_r,
        mean_g,
        mean_b,
        std_r,
        std_g,
        std_b,
        mean_h,
        mean_s,
        mean_v,
        std_h,
        std_s,
        std_v,
        brightness,
        contrast_simple,
        mean_int,
        std_int,
        min_int,
        max_int,
        range_int,
        skewness_val,
        kurtosis_val,
        entropy_val,
        contrast,
        dissimilarity,
        homogeneity,
        energy,
        correlation,
        ASM,
        edge_mean,
        edge_std,
        edge_density,
        contour_area,
        contour_perimeter,
        compactness,
    ]
    feats.extend(lbp_hist.tolist())
    feats.extend(color_hist.tolist())
    feats.extend(hog_feats.tolist())

    return np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
def extract_superpixel_features(img_bgr: np.ndarray, segments: np.ndarray):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_float = img_as_float(img_rgb)
    lab = rgb2lab(img_float)
    hsv = rgb2hsv(img_float)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, LBP_POINTS, LBP_RADIUS, method="uniform")
    edges = cv2.Canny(gray, 100, 200)
    ids = np.unique(segments)
    features = []
    masks = []
    pixel_counts = []
    for seg_id in ids:
        mask = segments == seg_id
        pixels = int(mask.sum())
        if pixels < SUPERPIXEL_MIN_PIXELS:
            continue
        masks.append(mask)
        pixel_counts.append(pixels)
        rows, cols = np.nonzero(mask)
        area_ratio = pixels / segments.size
        row_mean = rows.mean() / segments.shape[0]
        col_mean = cols.mean() / segments.shape[1]
        lab_vals = lab[mask]
        hsv_vals = hsv[mask]
        gray_vals = gray[mask]
        edge_density = float(np.count_nonzero(edges[mask])) / pixels
        lbp_vals = lbp[mask]
        lbp_hist = np.histogram(lbp_vals, bins=LBP_BINS, range=(0, LBP_BINS), density=True)[0].astype(np.float32)
        r0, r1 = rows.min(), rows.max() + 1
        c0, c1 = cols.min(), cols.max() + 1
        roi = gray[r0:r1, c0:c1]
        roi_mask = mask[r0:r1, c0:c1]
        roi_masked = roi.copy()
        roi_masked[~roi_mask] = 0
        roi_resized = cv2.resize(roi_masked, (48, 48), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
        hog_vec = hog(roi_resized, orientations=8, pixels_per_cell=(12, 12), cells_per_block=(1, 1), feature_vector=True).astype(np.float32)
        feat = np.concatenate([
            lab_vals.mean(axis=0).astype(np.float32),
            lab_vals.std(axis=0).astype(np.float32),
            hsv_vals.mean(axis=0).astype(np.float32),
            hsv_vals.std(axis=0).astype(np.float32),
            np.array([
                gray_vals.mean(),
                gray_vals.std(),
                area_ratio,
                row_mean,
                col_mean,
                edge_density,
            ], dtype=np.float32),
            lbp_hist,
            hog_vec,
        ])
        features.append(feat)
    if not features:
        return np.empty((0, 0), dtype=np.float32), [], []
    return np.asarray(features, dtype=np.float32), masks, pixel_counts


def compose_mask(masks, indices):
    if not masks:
        raise ValueError("No masks available.")
    shape = masks[0].shape
    keep = np.zeros(shape, dtype=bool)
    for idx in indices:
        keep |= masks[idx]
    return keep


def mask_image(original_img, masks, cluster_members, clusters):
    if not clusters:
        return np.zeros_like(original_img), np.zeros_like(masks[0], dtype=bool)
    indices = list(itertools.chain.from_iterable(cluster_members[c] for c in clusters))
    keep_mask = compose_mask(masks, indices)
    result = np.zeros_like(original_img)
    result[keep_mask] = original_img[keep_mask]
    return result, keep_mask


def true_class_prob(model, img_bgr, label_idx):
    feats = compute_global_features_from_bgr(img_bgr)
    probs = model.predict_proba(feats.reshape(1, -1))[0]
    return float(probs[label_idx])


def select_k(superpixel_features, pixel_counts):
    if len(superpixel_features) <= max(MIN_IMPORTANT_CLUSTERS, 3):
        return None
    scaler = StandardScaler()
    scaled = scaler.fit_transform(superpixel_features)
    candidates = []
    total_pixels = float(sum(pixel_counts))
    for k in K_RANGE:
        if len(scaled) <= k:
            continue
        model = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = model.fit_predict(scaled)
        if len(np.unique(labels)) < 2:
            continue
        cluster_pixels = np.zeros(k, dtype=float)
        for idx, lbl in enumerate(labels):
            cluster_pixels[lbl] += pixel_counts[idx]
        if total_pixels and (cluster_pixels / total_pixels).min() < 0.05:
            continue
        sil = silhouette_score(scaled, labels)
        ch = calinski_harabasz_score(scaled, labels)
        candidates.append({
            "k": k,
            "model": model,
            "labels": labels,
            "silhouette": sil,
            "calinski": ch,
        })
    if not candidates:
        return None
    best_sil = max(c["silhouette"] for c in candidates)
    filtered = [c for c in candidates if c["silhouette"] >= best_sil - SILHOUETTE_TOL]
    return max(filtered, key=lambda c: c["calinski"])


def prune_clusters(model, original_img, masks, cluster_labels, pixel_counts, label_idx):
    cluster_members = {}
    for idx, cluster_id in enumerate(cluster_labels):
        cluster_members.setdefault(int(cluster_id), []).append(idx)
    current_clusters = set(cluster_members.keys())
    initial_prob = true_class_prob(model, original_img, label_idx)
    current_prob = initial_prob
    removal_log = []
    while len(current_clusters) > MIN_IMPORTANT_CLUSTERS:
        best_cluster = None
        best_delta = None
        best_prob = None
        for cluster_id in current_clusters:
            candidate_clusters = current_clusters - {cluster_id}
            candidate_image, _ = mask_image(original_img, masks, cluster_members, candidate_clusters)
            prob = true_class_prob(model, candidate_image, label_idx)
            delta = current_prob - prob
            if best_delta is None or delta < best_delta:
                best_cluster = cluster_id
                best_delta = delta
                best_prob = prob
        if best_cluster is None or best_prob is None:
            break
        removal_log.append({
            "removed_cluster": int(best_cluster),
            "prob_after_removal": float(best_prob),
            "delta": float(best_delta),
        })
        current_clusters.remove(best_cluster)
        current_prob = best_prob
    keep_indices = list(itertools.chain.from_iterable(cluster_members[c] for c in current_clusters))
    if keep_indices:
        final_mask = compose_mask(masks, keep_indices)
    else:
        final_mask = np.zeros_like(masks[0], dtype=bool)
    final_image = np.zeros_like(original_img)
    final_image[final_mask] = original_img[final_mask]
    total_pixels = float(sum(pixel_counts))
    kept_pixels = sum(pixel_counts[idx] for idx in keep_indices)
    area_ratio = kept_pixels / total_pixels if total_pixels else 0.0
    return {
        "initial_prob": float(initial_prob),
        "final_prob": float(current_prob),
        "final_image": final_image,
        "final_mask": final_mask,
        "kept_clusters": sorted(int(c) for c in current_clusters),
        "removed_clusters": [entry["removed_cluster"] for entry in removal_log],
        "removal_log": removal_log,
        "kept_area_ratio": float(area_ratio),
    }

In [ ]:
summary_records = []
error_records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    class_name = row["class"]
    filename = row["filename"]
    label_idx = int(row[target_col])
    src_path = DATASET_ROOT / class_name / filename
    if not src_path.exists():
        error_records.append({"class": class_name, "filename": filename, "error": "missing source"})
        continue
    img_bgr = cv2.imread(str(src_path))
    if img_bgr is None:
        error_records.append({"class": class_name, "filename": filename, "error": "failed to load"})
        continue
    pixels = img_bgr.shape[0] * img_bgr.shape[1]
    adaptive_segments = int(np.clip(pixels / 2500, 80, 220))
    n_segments = max(SLIC_SEGMENTS, adaptive_segments)
    segments = slic(
        img_as_float(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)),
        n_segments=n_segments,
        compactness=SLIC_COMPACTNESS,
        sigma=SLIC_SIGMA,
        start_label=0,
        channel_axis=-1,
    )
    superpixel_features, masks, pixel_counts = extract_superpixel_features(img_bgr, segments)
    if superpixel_features.size == 0 or not masks:
        error_records.append({"class": class_name, "filename": filename, "error": "no superpixels"})
        continue
    selection = select_k(superpixel_features, pixel_counts)
    if selection is None:
        error_records.append({"class": class_name, "filename": filename, "error": "k selection failed"})
        continue
    cluster_labels = selection["labels"]
    prune_result = prune_clusters(prob_pipeline, img_bgr, masks, cluster_labels, pixel_counts, label_idx)
    dest_dir = OUTPUT_ROOT / class_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    saved = cv2.imwrite(str(dest_dir / filename), prune_result["final_image"])
    if not saved:
        error_records.append({"class": class_name, "filename": filename, "error": "save failed"})
        continue
    summary_records.append({
        "class": class_name,
        "filename": filename,
        "k_selected": selection["k"],
        "silhouette": float(selection["silhouette"]),
        "calinski": float(selection["calinski"]),
        "clusters_before": int(len(set(cluster_labels))),
        "clusters_after": int(len(prune_result["kept_clusters"])),
        "initial_prob": prune_result["initial_prob"],
        "final_prob": prune_result["final_prob"],
        "kept_area_ratio": prune_result["kept_area_ratio"],
        "kept_clusters": ";".join(map(str, prune_result["kept_clusters"])),
        "removed_clusters": ";".join(map(str, prune_result["removed_clusters"])),
        "removal_steps": len(prune_result["removed_clusters"]),
        "removal_log": json.dumps(prune_result["removal_log"], ensure_ascii=False),
    })

In [ ]:
summary_df = pd.DataFrame(summary_records)
summary_path = Path("phase5_region_summary.csv")
summary_df.to_csv(summary_path, index=False)
summary_df.head()

In [ ]:
errors_df = pd.DataFrame(error_records)
errors_path = Path("phase5_region_errors.csv")
errors_df.to_csv(errors_path, index=False)
errors_df.head()